In [18]:
import pandas as pd
from pulp import *

def risolvi_ottimizzazione_finale(filepath, limiti_bacche, sapori_target):
    """
    limiti_bacche: dizionario { 'NameBacca': max_unita }. 
                   Usa 0 per escludere completamente una bacca.
    """
    # 1. Caricamento dati
    df = pd.read_csv(filepath, sep=';')
    
    # 2. Inizializzazione Problema
    prob = LpProblem("Ottimizzazione_Ciambelle_Vincolata", LpMaximize)
    
    # 3. Variabili con limiti dinamici
    bacca_vars = {}
    for i, row in df.iterrows():
        Name = row['Name']
        # Se la bacca è nel dizionario filtri, usa quel limite, altrimenti 8
        limite_max = limiti_bacche.get(Name, 8)
        
        # Creiamo la variabile per questa specifica bacca
        bacca_vars[i] = LpVariable(f"B_{i}", lowBound=0, upBound=limite_max, cat='Integer')
    
    # Vincolo: totale 8 bacche nella ricetta
    prob += lpSum([bacca_vars[i] for i in df.index]) == 8
    
    # 4. Somme e Calcoli
    tutti_i_sapori = ['Sweet', 'Spicy', 'Sour', 'Bitter', 'Fresh']
    sapore_tot_assoluto = lpSum([bacca_vars[i] * df.loc[i, s] for i in df.index for s in tutti_i_sapori])
    tot_livello = lpSum([bacca_vars[i] * df.loc[i, 'Level Boost'] for i in df.index])
    tot_calorie = lpSum([bacca_vars[i] * df.loc[i, 'Calories'] for i in df.index])
    
    # 5. Vincolo di Parità (Bilanciamento Sapori Target)
    if len(sapori_target) > 1:
        for sapore in sapori_target[1:]:
            prob += lpSum([bacca_vars[i] * df.loc[i, sapori_target[0]] for i in df.index]) == \
                    lpSum([bacca_vars[i] * df.loc[i, sapore] for i in df.index])
            
    somma_sapori_scelti = lpSum([bacca_vars[i] * df.loc[i, s] for i in df.index for s in sapori_target])

    # 6. Logica Moltiplicatore (Scaglioni 120)
    scaglioni = [1, 2, 3, 4, 5]
    b = LpVariable.dicts("step", scaglioni, cat='Binary')
    for s in scaglioni:
        prob += sapore_tot_assoluto >= (s * 120) * b[s]
        prob += sapore_tot_assoluto <= (s * 120 - 1) + 10000 * b[s]

    bonus_totale = lpSum([b[s] * 0.1 for s in scaglioni])

    # 7. Funzione Obiettivo (Priorità Sapori > Bonus/Livello/Calorie)
    prob += (somma_sapori_scelti * 1000) + (tot_livello + tot_calorie) + (bonus_totale * 100)

    # Risoluzione
    prob.solve(PULP_CBC_CMD(msg=0))

    # 8. Output Risultati
    if LpStatus[prob.status] == 'Optimal':
        print(f"--- Ricetta Trovata ---")
        for i in df.index:
            quantita = value(bacca_vars[i])
            if quantita > 0:
                print(f"{int(quantita)}x {df.loc[i, 'Name']}")
        
        # Riepilogo valori per verifica parità
        for s in sapori_target:
            val = sum(value(bacca_vars[i]) * df.loc[i, s] for i in df.index)
            print(f"Totale {s}: {val}")
    else:
        print("Soluzione non trovata con questi vincoli di parità e limiti.")


In [42]:
a = pd.read_csv('./data/donut/bacche.csv', sep=';')
a = list(a['Name'])
miei_filtri = dict(zip(a, [2]*len(a)))

In [43]:
trovata = False
for i in miei_filtri:
    if i == 'Hyper Rindo Berry':
        trovata = True
    if trovata: miei_filtri[i] = 0

In [45]:
miei_filtri['Hyper Occa Berry'] = 0

In [46]:

risolvi_ottimizzazione_finale('./data/donut/bacche.csv', miei_filtri, ['Sour', 'Sweet'])

--- Ricetta Trovata ---
2x Hyper Pecha Berry
1x Hyper Lum Berry
2x Hyper Qualot Berry
1x Hyper Passho Berry
2x Hyper Wacan Berry
Totale Sour: 195.0
Totale Sweet: 195.0
